# QDGrasp Phase 3.4.3 — CUDA gate for the two active hands

Scope is **LEAP Hand and Wonik Allegro**. `ADR-0008` pauses the Shadow Hand, so
nothing here is three-hand coverage and a missing Shadow result reads
`paused_by_ADR-0008` — never `pass`, `zero`, `unsupported` or a bare `not_run`.

This notebook produces GPU evidence. It does **not** close the phase: closure
needs the completeness manifest with zero open required items and an independent
review, neither of which a benchmark can supply.

| stage | what a failure means |
| --- | --- |
| host check | a CPU kernel is not CUDA evidence (`ADR-0006`) |
| prior gates | a regression in the foundation, reported as such |
| Warp matrix | no clean version → the GPU gate stays **blocked** |
| capability | the build cannot read the contact fields the budget needs |
| parity | the GPU and the CPU oracle disagree about the same world |
| performance | below 2x, or above the 14 GiB device budget |

Dropping the sanitizer, filtering out bad worlds after a rollout, or lowering a
threshold are all *not* accepted resolutions.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

CODE_REVISION = "997c7be572c6dba87de758dea77f81bda77aa9d8"
MENAGERIE_REVISION = "da76818e269b82289eba39808e2fb91d679d6994"
REPO_URL = "https://github.com/ninicom/qdgrasp.git"
REPO_DIR = Path("/tmp/qdgrasp_repo")
ASSETS_DIR = Path("/tmp/robot-assets/mujoco-menagerie")
# /kaggle/working is the only directory Kaggle persists as a kernel
# output. Writing to /tmp produced a run whose evidence could not be
# downloaded, which is a packet nobody can review.
EVIDENCE_DIR = Path("/kaggle/working/phase3_4_3_evidence")
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

assert sys.version_info >= (3, 11), f"Python >=3.11 required, got {sys.version}"
os.environ.update(
    QDGRASP_ROBOT_ASSETS_ROOT="/tmp/robot-assets",
    MUJOCO_GL="egl",
    PYTHONHASHSEED="0",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
)

# The full pinned set, not a subset: importing qdgrasp pulls in the engine
# runner, which needs lightning. Trimming the list to what the gate "obviously"
# uses is how the first run died in the prior-gates cell.
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
    "lightning==2.6.5", "mujoco==3.12.0", "numpy==2.4.6", "scipy==1.17.1",
    "trimesh==4.12.2", "safetensors==0.8.0", "pydantic==2.13.4", "PyYAML==6.0.3",
    "einops==0.8.2", "rich==14.3.4", "typer==0.27.1", "torchmetrics==1.9.0",
    "Pillow==12.1.1", "pytest==9.1.1", "nvidia-ml-py",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--force-reinstall",
    f"git+{REPO_URL}@{CODE_REVISION}",
], check=True)

for directory, url, revision in (
    (REPO_DIR, REPO_URL, CODE_REVISION),
    (ASSETS_DIR, "https://github.com/google-deepmind/mujoco_menagerie.git", MENAGERIE_REVISION),
):
    if not directory.exists():
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", url, str(directory)], check=True)
    subprocess.run(["git", "-C", str(directory), "fetch", "--depth", "1", "origin", revision], check=True)
    subprocess.run(["git", "-C", str(directory), "checkout", "--detach", revision], check=True)
    actual = subprocess.check_output(["git", "-C", str(directory), "rev-parse", "HEAD"], text=True).strip()
    assert actual == revision, (directory, actual, revision)

print("Pinned QDGrasp revision:", CODE_REVISION)
print("Pinned Menagerie revision:", MENAGERIE_REVISION)


## 1. Refuse a CPU host

A CPU fallback is never admissible as CUDA evidence
(`docs/decisions/0006-cuda-hardware-required.md`). This cell fails the run rather
than continuing on CPU.


In [ ]:
import json
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "no CUDA device: this notebook must run on a GPU kernel"
props = torch.cuda.get_device_properties(0)
FINGERPRINT = {
    "gpu": torch.cuda.get_device_name(0),
    "capability": f"{props.major}.{props.minor}",
    "vram_gib": round(props.total_memory / (1024 ** 3), 3),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "driver": subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip(),
    "python": sys.version.split()[0],
}
print(json.dumps(FINGERPRINT, indent=2, sort_keys=True))


## 2. The prior CUDA gates still pass

Section 10 of `ROADMAP-P3.4-001` requires the Phase 1 CUDA smoke and the Phase 2
active-hand FK parity to be re-run before any Phase 3.4.3 measurement, so a
regression in the foundation is never reported as a Phase 3.4.3 result.


In [ ]:
import subprocess
import sys

for script, out in (
    ("scripts/phase1_cuda_smoke.py", "/kaggle/working/phase3_4_3_evidence/phase1_cuda.json"),
    ("scripts/phase2_cuda_fk_parity.py", "/kaggle/working/phase3_4_3_evidence/phase2_cuda.json"),
):
    print("=" * 70)
    print("running", script)
    completed = subprocess.run(
        [sys.executable, script, "--out", out],
        cwd="/tmp/qdgrasp_repo", capture_output=True, text=True,
    )
    print(completed.stdout[-3000:])
    print(completed.stderr[-2000:], file=sys.stderr)
    assert completed.returncode == 0, f"{script} failed with {completed.returncode}"


## 3. MuJoCo Warp compatibility matrix

`REV-20260827-010` isolated an uninitialised-read defect in MuJoCo Warp 1.16.0
and showed it reproduces with no QDGrasp code in the call path. The plan allows
exactly two resolutions: a newer pinned version that is clean, or a blocked GPU
gate. Removing the sanitizer, or discarding the worlds it flags, is not one of
them.

This cell tries 3 versions under `compute-sanitizer --tool
initcheck` on a small reproducer and reports which, if any, is clean. Small on
purpose: the sanitizer costs one to two orders of magnitude, and this is a
diagnostic, never performance evidence.


In [ ]:
import json
import shutil
import subprocess
import sys

WARP_MATRIX = ['mujoco-warp==3.12.0', 'mujoco-warp==3.11.0', 'mujoco-warp==3.10.0.3']
sanitizer = shutil.which("compute-sanitizer")
print("compute-sanitizer:", sanitizer or "NOT FOUND")

# The reproducer from REV-20260827-010 V-003, not a toy scene. The defect was
# isolated on the LEAP hand model with its meshes; a three-geom scene may never
# reach the kernel in question, so a clean result there would say nothing about
# the defect this matrix exists to re-test.
PROBE = (
    "import mujoco, mujoco_warp\n"
    "from qdgrasp.dataset.pipeline.generated_reachable import "
    "build_generated_reachable_object as f\n"
    "from qdgrasp.dataset.pipeline.validators.mujoco_rollout import "
    "build_rollout_scene_model as b\n"
    "from qdgrasp.robot.spec import RobotSpec, resolve_robot_asset\n"
    "s = RobotSpec.from_config('leap_hand.yaml', sample_anchors=False)\n"
    "x = f('leap_hand')\n"
    "m = b(resolve_robot_asset(s.config.source_asset), x.collision_geoms,"
    " object_pos=x.object_pos, object_mass=x.mass)\n"
    "print('model ngeom', m.ngeom, 'nq', m.nq, 'nu', m.nu)\n"
    "d = mujoco.MjData(m); mujoco.mj_forward(m, d)\n"
    "wm = mujoco_warp.put_model(m); wd = mujoco_warp.put_data(m, d, nworld=4)\n"
    "for _ in range(8):\n"
    "    mujoco_warp.step(wm, wd)\n"
    "print('stepped 8 times over 4 worlds')\n"
)

WARP_MATRIX_RESULT = {}
for pin in WARP_MATRIX:
    print("=" * 70)
    print("trying", pin)
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "warp-lang", pin],
        capture_output=True, text=True,
    )
    if install.returncode != 0:
        WARP_MATRIX_RESULT[pin] = {
            "status": "install_failed",
            "detail": (install.stderr or install.stdout)[-600:],
        }
        print("install failed:", (install.stderr or install.stdout).strip().splitlines()[-1][:200])
        continue
    if not sanitizer:
        WARP_MATRIX_RESULT[pin] = {"status": "sanitizer_unavailable"}
        continue
    run = subprocess.run(
        [sanitizer, "--tool", "initcheck", "--error-exitcode", "0", "--print-limit", "8",
         sys.executable, "-c", PROBE],
        cwd="/tmp/qdgrasp_repo", capture_output=True, text=True, timeout=5400,
    )
    text = run.stdout + run.stderr
    records = [ln.strip() for ln in text.splitlines() if ln.lstrip().startswith("=========")]
    summary = [ln for ln in records if "ERROR SUMMARY" in ln]
    errors = [ln for ln in records if "error" in ln.lower() and "0 errors" not in ln.lower()]
    stepped = "stepped 8 times over 4 worlds" in run.stdout

    # "Clean" needs positive proof, not the absence of an error line: a probe
    # that died before the first instrumented call also prints no errors, and
    # that is the reading that produced a false clean in the previous run.
    if not stepped:
        status = "probe_did_not_run"
    elif errors:
        status = "uninitialized_reads"
    elif summary and "0 errors" in summary[0]:
        status = "clean"
    else:
        status = "inconclusive_no_error_summary"

    WARP_MATRIX_RESULT[pin] = {
        "status": status,
        "probe_stepped": stepped,
        "sanitizer_lines": len(records),
        "error_summary": summary[:2],
        "first_errors": errors[:5],
        "verbatim_tail": records[-8:],
        "probe_stdout_tail": run.stdout[-400:],
    }
    print(status, f"({len(records)} sanitizer lines, stepped={stepped})")
    for ln in summary[:2]:
        print("   ", ln[:160])

try:
    import warp as _warp
    _warp_version = getattr(_warp, "__version__", "unknown")
except Exception as _exc:
    _warp_version = f"unavailable: {type(_exc).__name__}"

# The runtime version goes beside the results, not among them: mixing a scalar
# into a dict of per-pin dicts is what broke the previous run.
WARP_MATRIX_REPORT = {"warp_lang_version": _warp_version, "pins": WARP_MATRIX_RESULT}
json.dump(
    WARP_MATRIX_REPORT,
    open("/kaggle/working/phase3_4_3_evidence/warp_matrix.json", "w"),
    indent=2, sort_keys=True,
)
CLEAN = [pin for pin, r in WARP_MATRIX_RESULT.items() if r["status"] == "clean"]
print()
print("clean versions:", CLEAN or "none — the GPU gate stays BLOCKED")


## 3b. Is it the linesearch, or the backend?

The version matrix answers "is any release clean". It cannot tell us *why* they
are not. The defect names `_linesearch_iterative_kernel`, which only some solver
configurations reach, so this cell holds the version fixed at the newest pin and
varies the solver instead.

A clean result here would be worth having: it would identify the defect precisely
and give the gate a documented configuration to run under, which is a fallback
the plan's section 3.7 allows. It would not be a way of hiding the sanitizer --
the same positive-proof rule applies, and a variant that fails to step is
reported as such rather than counted clean.


In [ ]:
SOLVER_VARIANTS = (('baseline', '', ''), ('ls_parallel', '', 'wm.opt.ls_parallel = True\n'), ('solver_cg', 'm.opt.solver = mujoco.mjtSolver.mjSOL_CG\n', ''), ('ls_iterations_1', 'm.opt.ls_iterations = 1\n', ''))
SOLVER_RESULT = {}

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "warp-lang", WARP_MATRIX[0]],
    capture_output=True, text=True,
)

for name, pre, post in SOLVER_VARIANTS:
    body = PROBE.replace(
        "wm = mujoco_warp.put_model(m)",
        pre + "wm = mujoco_warp.put_model(m)",
    ).replace(
        "for _ in range(8):",
        post + "for _ in range(8):",
    )
    open("/tmp/solver_probe.py", "w").write(body)
    print("=" * 70)
    print("solver variant:", name)
    if not sanitizer:
        SOLVER_RESULT[name] = {"status": "sanitizer_unavailable"}
        continue
    run = subprocess.run(
        [sanitizer, "--tool", "initcheck", sys.executable, "/tmp/solver_probe.py"],
        capture_output=True, text=True, timeout=1800,
    )
    blob = run.stdout + run.stderr
    stepped = "stepped 8 times over 4 worlds" in blob
    summary = [ln for ln in blob.splitlines() if "ERROR SUMMARY:" in ln]
    if not stepped:
        status = "probe_did_not_run"
    elif not summary:
        status = "inconclusive_no_error_summary"
    elif summary[-1].strip().endswith("0 errors"):
        status = "clean"
    else:
        status = "errors"
    SOLVER_RESULT[name] = {
        "status": status,
        "probe_stepped": stepped,
        "error_summary": summary[-1].strip() if summary else "",
        "verbatim_tail": blob[-800:],
    }
    print(" ", status, "|", summary[-1].strip() if summary else "(no summary line)")

SOLVER_REPORT = {"pin": WARP_MATRIX[0], "variants": SOLVER_RESULT}
json.dump(
    SOLVER_REPORT,
    open("/kaggle/working/phase3_4_3_evidence/solver_variants.json", "w"),
    indent=2, sort_keys=True,
)
CLEAN_SOLVER = [n for n, r in SOLVER_RESULT.items() if r["status"] == "clean"]
print()
print("clean solver variants:", CLEAN_SOLVER or "none")


## 3c. Does the sanctioned fallback exist?

Section 3.7 allows exactly two resolutions when no MJWarp version is clean: a
fallback backend that passes capability and parity on its own evidence, or a
blocked GPU gate. We have been asserting the first is "new work needing its own
plan" without checking whether it is possible at all, which is an assumption
wearing the clothes of a finding.

MJX is the other batched GPU backend for MuJoCo. The contract's capability gate
needs one thing above all: per-contact force, readable. If MJX cannot supply it,
the fallback branch is closed too and the only remaining path is upstream --
which is worth knowing precisely, because it is the difference between "two
options" and "none of ours".

This installs MJX and asks. It is a capability probe, never performance evidence.


In [ ]:
MJX_RESULT = {"attempted": True}
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "mujoco-mjx"],
    capture_output=True, text=True,
)
MJX_RESULT["install_ok"] = install.returncode == 0
if install.returncode != 0:
    MJX_RESULT["detail"] = (install.stderr or install.stdout)[-600:]
    print("mjx install failed")
else:
    probe = (
        "import json, mujoco\n"
        "from mujoco import mjx\n"
        "import jax\n"
        "from qdgrasp.dataset.pipeline.generated_reachable import build_generated_reachable_object as f\n"
        "from qdgrasp.dataset.pipeline.validators.mujoco_rollout import build_rollout_scene_model as b\n"
        "from qdgrasp.robot.spec import RobotSpec, resolve_robot_asset\n"
        "s = RobotSpec.from_config('leap_hand.yaml', sample_anchors=False)\n"
        "x = f('leap_hand')\n"
        "m = b(resolve_robot_asset(s.config.source_asset), x.collision_geoms, object_pos=x.object_pos, object_mass=x.mass)\n"
        "mx = mjx.put_model(m)\n"
        "d = mujoco.MjData(m); mujoco.mj_forward(m, d)\n"
        "dx = mjx.put_data(m, d)\n"
        "dx = jax.jit(mjx.step)(mx, dx)\n"
        "cattrs = sorted(a for a in dir(dx.contact) if not a.startswith('_'))\n"
        "out = dict(stepped=True, jax_devices=str(jax.devices()), has_efc_force=hasattr(dx, 'efc_force'), contact_attrs=cattrs, ncon=int(dx.ncon) if hasattr(dx, 'ncon') else -1)\n"
        "print('MJX_PROBE_JSON ' + json.dumps(out))\n"
    )
    open("/tmp/mjx_probe.py", "w").write(probe)
    run = subprocess.run(
        [sys.executable, "/tmp/mjx_probe.py"], capture_output=True, text=True, timeout=1800,
    )
    blob = run.stdout + run.stderr
    line = [ln for ln in blob.splitlines() if ln.startswith("MJX_PROBE_JSON ")]
    if line:
        MJX_RESULT["probe"] = json.loads(line[0][len("MJX_PROBE_JSON "):])
        MJX_RESULT["status"] = "stepped"
    else:
        MJX_RESULT["status"] = "probe_did_not_run"
        MJX_RESULT["verbatim_tail"] = blob[-1200:]
    print(json.dumps(MJX_RESULT, indent=2)[:1500])

json.dump(
    MJX_RESULT,
    open("/kaggle/working/phase3_4_3_evidence/mjx_capability.json", "w"),
    indent=2, sort_keys=True,
)


## 4. Dry run: what the gate will cost

`C07.1` requires the resource estimate to be printed before the run, not
discovered when it OOMs halfway through.


In [ ]:
import subprocess
import sys

dry = subprocess.run(
    [sys.executable, "scripts/check_phase3_4_3_cuda.py", "--dry-run", "--worlds", "1024"],
    cwd="/tmp/qdgrasp_repo", capture_output=True, text=True,
)
print(dry.stdout)
assert dry.returncode == 0, dry.stderr


## 5. Sanitizer on the gate's own workload

Zero invalid reads is a gate criterion, not a nice-to-have. Bounded worlds and
horizon because the sanitizer is slow; this is a diagnostic and never
performance evidence.


In [ ]:
import json
import shutil
import subprocess
import sys

sanitizer = shutil.which("compute-sanitizer")
SANITIZER_RESULT = {"tool_available": bool(sanitizer)}
if not sanitizer:
    print("compute-sanitizer unavailable; the question stays open rather than guessed at.")
else:
    for tool in ("racecheck", "initcheck"):
        run = subprocess.run(
            [sanitizer, "--tool", tool, "--error-exitcode", "0", "--print-limit", "40",
             sys.executable, "scripts/phase3_4_1_sanitizer.py", "--worlds", "4", "--horizon", "8"],
            cwd="/tmp/qdgrasp_repo", capture_output=True, text=True, timeout=5400,
        )
        text = run.stdout + run.stderr
        records = [ln.strip() for ln in text.splitlines() if ln.lstrip().startswith("=========")]
        SANITIZER_RESULT[tool] = {"lines": len(records), "head": [ln[:190] for ln in records[:20]]}
        json.dump(
            SANITIZER_RESULT,
            open("/kaggle/working/phase3_4_3_evidence/sanitizer.json", "w"),
            indent=2, sort_keys=True,
        )
        print("=" * 70)
        print(tool, f"-- {len(records)} report lines")
        for ln in records[:20]:
            print(ln[:190])


## 6. The Phase 3.4.3 CUDA gate

Capability, three-tier parity and performance, in that order, with a checkpoint
so a wall-clock kill does not throw away the stages that already finished.
It reads the sanitizer report written above: parity and performance are
necessary but not sufficient, and a backend whose kernels read uninitialised
memory produces numbers nobody should rank (G08.7). The
deadline guard flushes the ledger before the Kaggle session ends.

If the Warp matrix found no clean version, run this anyway and record the
result: a blocked gate with measured evidence is a legitimate outcome, and a
skipped gate is not.


In [ ]:
import json
import subprocess
import sys

gate = subprocess.run(
    [sys.executable, "scripts/check_phase3_4_3_cuda.py",
     "--device", "cuda:0", "--worlds", "1024", "--runs", "3",
     "--evidence", "/kaggle/working/phase3_4_3_evidence/cuda-gate.json",
     "--checkpoint", "/kaggle/working/phase3_4_3_evidence/cuda-gate.checkpoint.json",
     "--sanitizer-report", "/kaggle/working/phase3_4_3_evidence/sanitizer.json",
     "--deadline-seconds", "24000"],
    cwd="/tmp/qdgrasp_repo", capture_output=True, text=True,
)
print(gate.stdout[-8000:])
print(gate.stderr[-3000:], file=sys.stderr)
print("gate exit:", gate.returncode)

# A gate that refused to run still has to leave a readable result. If it exited
# before writing evidence -- a configuration error, a missing dependency -- the
# reason is what matters, and turning that into a KeyError here would hide it.
from pathlib import Path as _Path

_evidence = _Path("/kaggle/working/phase3_4_3_evidence/cuda-gate.json")
if _evidence.is_file():
    GATE = json.loads(_evidence.read_text(encoding="utf-8"))
else:
    GATE = {
        "verdict": "NO_EVIDENCE",
        "gate_exit": gate.returncode,
        "stdout_tail": gate.stdout[-2000:],
        "stderr_tail": gate.stderr[-2000:],
    }
    _evidence.parent.mkdir(parents=True, exist_ok=True)
    _evidence.write_text(json.dumps(GATE, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("VERDICT:", GATE["verdict"])


## 7. Environment fingerprint and evidence hashes

Everything a reviewer needs to tell whether two runs are comparable, and nothing
a notebook should not carry: no credentials, no private paths, no mutable
notebook name used as a pin.


In [ ]:
import hashlib
import json
from pathlib import Path

PACKET = {
    "schema": "qdgrasp/evidence/phase3.4.3-kaggle/v1",
    "commit": CODE_REVISION,
    "menagerie": MENAGERIE_REVISION,
    "environment": FINGERPRINT,
    "warp_matrix": WARP_MATRIX_REPORT,
    "sanitizer": SANITIZER_RESULT,
    "gate_verdict": GATE["verdict"],
    "artifact_hashes": {},
}
for path in sorted(Path("/kaggle/working/phase3_4_3_evidence").glob("*.json")):
    PACKET["artifact_hashes"][path.name] = hashlib.sha256(path.read_bytes()).hexdigest()

out = Path("/kaggle/working/phase3_4_3_evidence/packet.json")
out.write_text(json.dumps(PACKET, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(PACKET, indent=2, sort_keys=True))
print()
print("packet sha256:", hashlib.sha256(out.read_bytes()).hexdigest())
print()
print("Download /kaggle/working/phase3_4_3_evidence/ and commit it under evidence/phase3_4_3/s10/.")
print("A PASS here is GPU evidence for two active hands. It is not phase closure,")
print("and it is not three-hand coverage.")
